# P5 · Đánh giá & tổng hợp — Bước 6–7–8

**Input (cell đầu tự nạp):** data/outputs/forecasts/*.parquet, common/segment_labels.parquet, data/processed/data_long.parquet

**Output (cell cuối tự lưu):** data/outputs/{metrics_by_model_segment,decision_table}.parquet

**Code mẫu tham khảo:** `notebooks/M5_reference.ipynb` phần **§7**.

> Chạy được **cell đầu** (thấy `✅ Input sẵn sàng`) = đủ input để làm phần của bạn. Code ở khu vực giữa. Cuối cùng chạy **cell lưu output**.

## Cell đầu — import + nạp input (chạy được = sẵn sàng)

In [ ]:
from pathlib import Path
def _need(p, hint):
    assert Path(p).exists(), f"❌ THIẾU INPUT: {p}\n   → {hint}"

import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
from scipy import stats
_need("common/config.yaml", "clone repo")
_need("common/segment_labels.parquet", "clone repo (P1 đã tạo)")
_need("data/processed/data_long.parquet", "chạy P1.ipynb hoặc tải data_long P1 gửi vào data/processed/")
CFG = yaml.safe_load(open("common/config.yaml", encoding="utf-8"))
Q = CFG["quantiles"]; H = CFG["horizon"]; SP = CFG["split"]
seg = pd.read_parquet("common/segment_labels.parquet")
windows = pd.read_csv("common/rolling_windows.csv")
long = pd.read_parquet("data/processed/data_long.parquet")
FCDIR = Path("data/outputs/forecasts")
forecasts = {p.stem: pd.read_parquet(p) for p in sorted(FCDIR.glob("*.parquet"))}
assert forecasts, "❌ Chưa có forecast nào trong data/outputs/forecasts/ — chờ P3/P4 chạy + push rồi git pull."
print("✅ Input sẵn sàng — forecast từ:", list(forecasts.keys()))

# ===== HỢP ĐỒNG LƯỚI (grid contract) — chặn so sánh lẫn quy mô =====
# Lưới chuẩn lấy từ common/: mọi forecast phải phủ ĐÚNG tập series × cửa sổ × quantile × H.
EXP_SERIES = set(long["series_id"].unique())
EXP_DINT = set()
for _, w in windows.iterrows():
    EXP_DINT |= set(range(int(w["test_start"]), int(w["test_end"]) + 1))
EXP_ROWS = len(windows) * len(EXP_SERIES) * H * len(Q)

def _grid_issues(df):
    iss = []
    if set(df["series_id"].unique()) != EXP_SERIES:
        iss.append(f"{df['series_id'].nunique()} series (cần {len(EXP_SERIES)})")
    if set(df["d_int"].unique()) != EXP_DINT:
        iss.append(f"{df['d_int'].nunique()} ngày test (cần {len(EXP_DINT)})")
    if sorted(df["quantile_level"].unique()) != sorted(Q):
        iss.append("sai quantile")
    return iss

_rows, forecasts_ok = [], {}
for name, df in forecasts.items():
    iss = _grid_issues(df)
    _rows.append({"model": name, "rows": len(df), "series": df["series_id"].nunique(),
                  "cửa sổ": int(df["d_int"].nunique() / H) if df["d_int"].nunique() % H == 0 else -1,
                  "trạng thái": "đúng lưới" if not iss else "LỆCH: " + "; ".join(iss)})
    if not iss:
        forecasts_ok[name] = df

coverage = pd.DataFrame(_rows)
print(coverage.to_string(index=False))
print(f"\nLưới chuẩn: {len(EXP_SERIES):,} series × {len(windows)} cửa sổ × {len(Q)} q × {H} ngày = {EXP_ROWS:,} dòng/model")
print(f"→ Dùng để đánh giá (forecasts_ok): {list(forecasts_ok.keys())}")
if len(forecasts_ok) < len(forecasts):
    print("⚠️  Có forecast LỆCH LƯỚI đã bị loại — chờ P3 chạy lại full-scale (lgbm + cổ điển).")
# LƯU Ý: phần đánh giá bên dưới hãy dùng `forecasts_ok` (không dùng `forecasts`) để tránh lẫn quy mô.

## Phần code chính — tự triển khai ở đây
(Thêm bao nhiêu cell tuỳ ý giữa cell input và cell lưu output.)

In [ ]:
# TODO: code phần việc của bạn ở đây


## Cell cuối — lưu output (điền biến theo comment rồi chạy)

In [ ]:
# ===== lưu output =====
# metrics_table (model×nhóm) và decision_table (bảng thắng theo nhóm)
Path("data/outputs").mkdir(parents=True, exist_ok=True)
metrics_table.to_parquet("data/outputs/metrics_by_model_segment.parquet", index=False)
decision_table.to_parquet("data/outputs/decision_table.parquet", index=False)
print("✅ Đã lưu metrics_by_model_segment + decision_table")